# Phase of the guide and the target in I2SB-Recon

The I2SB baseline (`sbunet_inv`) bridges from the PD guide to the PDFS target. Both are complex
coil combinations, and nothing aligns their phases:

- the target is combined under the operator's maps, calibrated on the PDFS measurement's ACS;
- the guide is combined under **its own** maps (`guide_maps = "own"`), calibrated on the PD scan;
- ESPIRiT fixes each set of maps' per-pixel phase by making the scan's **strongest coil** real
  (`physics/smaps.py`), and the two scans need not pick the same coil;
- the two sequences also differ in object phase (off-resonance, fat suppression).

The network does not see the guide (`cond_x1=False`), so it has to guess the target's phase. A guess
with the wrong phase disagrees with $y$ on the measured lines, and 7 CG iterations at R = 8 only
partly unfold that disagreement. That would put noise and aliasing into every post-CG image.

This notebook tests the chain in three parts:

1. **Maps.** Do the PD and PDFS maps of the same slice differ only by a per-pixel phase shared by
   all coils? If so, the two scans see the same coil profiles, and combining either scan under
   either set of maps changes only its phase.
2. **Guide against target.** How large is the phase difference between the bridge's two ends, how
   much of it is smooth, and how much does each alignment remove (the target's maps for the guide;
   Sljiva's `phase_correct` on both)?
3. **Oracle swap.** Along one sampler run, give CG the network's guess with the **true phase** (its
   own magnitude), or with the **true magnitude** (its own phase). If the true phase removes most of
   the residual and of the post-CG noise, phase is the error to fix, and retraining with aligned
   phases is worth it.

`PHASE_CORRECT = True` rebuilds the target, the guide and the operator with `phase_correct` on both
sets of maps. Only a network trained that way should be sampled like that; parts 1 and 2 do not
use the network.

**Where to run:** bigpurple, on a GPU node. The run directory and the paired knee data live there.

In [ ]:
%matplotlib inline
import os, pathlib, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from datasets.registry import build_loader
from operators import Sense
from operators.fourier import ifftc
from physics.mask import resolve_acs_lines
from physics.online_smaps import online_smaps
from sb.base import _per_sample, bridge_coeffs, forward_std, n_steps, space_indices
from sb.i2sb_recon import bridge_noise_scale, cg_data_consistency, predict_x0_recon
from scripts.eval_i2sb_recon_dc import load_run
from training.forward_op import fixed_val_subset
from training.i2sb_recon import _measure
from training.metrics import compute_metrics
from training.recon import _unpack, undersample_guide

In [ ]:
# ===================================== SETTINGS =====================================
RUN = "trained_nets/guided_recon/knee/s4_noise-vlow/sbunet_inv_R8_g1_tau0.19"
CKPT = "net_bestval.ckpt"
SLICE = 0                   # index into the run's fixed val subset
PHASE_CORRECT = None        # None: the run's own maps. True / False: force Sljiva's phase_correct
                            #   on BOTH the target's and the guide's maps (the target, the guide
                            #   and the operator are all rebuilt). Sample with True only a net
                            #   trained with it; parts 1-2 do not use the net.
KSPACE = "simulated"        # "simulated": y = E x0 + SIM_NOISE; "measured": the scan's own k-space
SIM_NOISE = 1e-3
VAL_NOISE = 0.0             # "measured" only: ADDED noise std
NFE = 20                    # part 3: network calls
DC_ITERS = 7                # part 3: CG iterations per prediction (their code: 7)
DC_LAM = 7e-5               # part 3: the CG weight (their code: 7e-5); None = no CG
SEED = 0                    # the bridge noise
SUPPORT = 0.1               # object support: above this fraction of each image's 99th
                            #   percentile, in BOTH scans; every statistic is taken on it
ACS_REF = 32                # part 1's yardstick: the SAME PDFS scan's maps from this many ACS
                            #   lines, against the operator's count
LP_SIGMA = 8.0              # px: the Gaussian low-pass that splits a phase difference into a
                            #   smooth part and the rest
COILS_SHOW = 4              # part 1: the strongest coils shown
SHOW = 5                    # part 3: steps shown, evenly spaced
DISPLAY_ROWS = 320          # centre crop of the readout axis for display; None = all

In [ ]:
run = load_run(RUN, CKPT)
net, bridge, i2, tr, dev = (run[k] for k in ("net", "bridge", "i2", "tr", "device"))
meas = dict(run["meas"])
kws = dict(meas.get("online_smaps_kws") or {})
if PHASE_CORRECT is not None:
    kws["phase_correct"] = bool(PHASE_CORRECT)
meas["online_smaps_kws"] = kws
if meas["x1_source"] != "guide":
    raise ValueError(f"this notebook needs the guide as the bridge start; x1 = {meas['x1_source']!r}")
if not meas.get("online_smaps"):
    raise ValueError("this notebook needs online maps (mri.online_smaps): paired data stores none")
if i2.get("posterior", "ddpm") != "ddpm" or i2.get("deterministic", False):
    raise ValueError("part 3 writes out the stochastic ddpm posterior")

vcfg = dict(run["cfg"]["data"]["val"], batch_size=1)
loader = fixed_val_subset(build_loader(vcfg, shuffle=False, drop_last=False),
                          tr.get("val_slices"), run["seed"])
batch = loader.dataset[SLICE]
batch = [t[None] if torch.is_tensor(t) else torch.as_tensor(t)[None] for t in batch]
gen = torch.Generator(device=dev).manual_seed(SEED)
x0, x1, y, E, sigma_n, organ, gt, guide = _measure(
    batch, net, dev, kspace_type=run["kspace_type"],
    noise_std=run["noise_std"] if VAL_NOISE is None else VAL_NOISE,
    generator=gen, guide_offset=0, **meas)
kspace, _, _, _, _, side = _unpack(batch)
if isinstance(side, dict):
    raise ValueError("one guide slice only (data.*.guide_slices = 1)")
kspace, side = kspace.to(dev), side.to(dev)


def find_smaps(op):
    for o in getattr(op, "ops", [op]):
        if hasattr(o, "smaps"):
            return o.smaps
    raise ValueError("no Sense operator in E")


S_op = find_smaps(E)                  # the operator's maps: the target is combined under these
kappa = bridge_noise_scale(x1, i2.get("tau_scale", "none"), i2.get("tau_ref"))
mr = tr.get("metric_range", 1.0)
print(f"run {RUN}\n  guide maps: {meas.get('guide_maps')!r}, guide_R = {meas.get('guide_R', 1)}, "
      f"maps {meas['online_smaps']} {kws}\n  image {tuple(x0.shape[-2:])}, "
      f"{kspace.shape[1]} coils, tau = {i2['tau']}")

In [ ]:
if KSPACE == "simulated":
    sampled = (y != 0)
    y = E.forward(x0)
    if SIM_NOISE:
        g_ = torch.Generator(device=dev).manual_seed(SEED + 7)
        y = y + SIM_NOISE * sampled * torch.randn(y.shape, dtype=y.dtype, device=y.device,
                                                  generator=g_)
    sigma_n = torch.full_like(sigma_n, float(SIM_NOISE))
    gt = x0
    print(f"simulated: y = E x0 + noise of std {SIM_NOISE}; ground truth = x0")
elif KSPACE != "measured":
    raise ValueError(f"KSPACE {KSPACE!r}: 'simulated' or 'measured'")


def crop(a):
    if DISPLAY_ROWS and a.shape[-2] > DISPLAY_ROWS:
        c = (a.shape[-2] - DISPLAY_ROWS) // 2
        a = a[..., c:c + DISPLAY_ROWS, :]
    return a


def show(a):
    a = a.detach().cpu()
    while a.dim() > 2:
        a = a[0]
    return crop(a).numpy()


def sgn(v):
    a = v.abs()
    return torch.where(a > 0, v / a.clamp_min(1e-30), torch.ones_like(v))


def q99(v):
    return float(torch.quantile(v.abs().flatten().float(), 0.99))


def masked_phase(ph, supp):
    # a phase map with the background blanked (NaN draws as the axes' background)
    out = show(ph).copy()
    out[~show(supp)] = np.nan
    return out

## 1. Do the PD and PDFS maps differ only in phase?

Both sets of maps are calibrated here with the operator's settings: the same ACS width, kernel and
thresholds. The phase normalization is left at plain ESPIRiT (`phase_correct` off), since
`phase_correct` changes the phase on purpose.

Write $S^{a}$ for the PDFS maps and $S^{b}$ for the PD maps, both $C$-vectors at each pixel $r$.
If $S^{b}(r) = e^{i\varphi(r)} S^{a}(r)$, with one phase per pixel shared by every coil, then:

- the **coherence** $\rho(r) = |\langle S^{a}(r), S^{b}(r)\rangle| / (\|S^{a}(r)\|\,\|S^{b}(r)\|)$ is 1;
- after rotating $S^{a}$ by the best per-pixel phase $\varphi(r) = \arg\langle S^{a}, S^{b}\rangle$,
  nothing is left: $\|S^{b} - e^{i\varphi}S^{a}\|$ is zero;
- every coil's own phase difference, minus $\varphi$, is zero.

Estimates are never exact, so the numbers need a yardstick: the **same** PDFS scan's maps from
`ACS_REF` ACS lines instead of the operator's count. A PD–PDFS difference at the yardstick's level
means the maps agree up to phase. A larger one means the coil profiles moved between the scans
(the patient moved, or the slices were paired off by one).

All numbers are energy-weighted over the object support of both scans.

In [ ]:
W = kspace.shape[-1]
method = meas["online_smaps"]
acs = resolve_acs_lines(W, meas["acs_lines"], meas.get("center_frac"))
full = torch.ones(1, 1, 1, W, device=dev)


def maps(k, mask=full, lines=acs, pc=False):
    return online_smaps(k, mask, method=method, **dict(kws, acs_lines=lines, phase_correct=pc))


g = undersample_guide(side, meas.get("guide_R", 1), meas.get("guide_acs_lines", 0),
                      meas.get("guide_mask_dist", "uniform"), 0)
gmask = (g != 0).any(dim=1, keepdim=True).any(dim=2, keepdim=True).to(g.real.dtype)
gacs = resolve_acs_lines(g.shape[-1], meas["acs_lines"], meas.get("center_frac"))
S_fs = maps(kspace)                                   # PDFS, plain ESPIRiT
S_fs_ref = maps(kspace, lines=ACS_REF)                # PDFS again, the yardstick's ACS width
S_pd = maps(g, gmask, lines=gacs)                     # PD, as _measure calibrates the guide
gk = ifftc(g)

supp = ((x0.abs() > SUPPORT * q99(x0)) & (Sense(S_pd).adjoint(gk).abs() >
        SUPPORT * q99(Sense(S_pd).adjoint(gk))))[:, 0]           # (B, H, W)


def compare_maps(Sa, Sb):
    ip = (Sa.conj() * Sb).sum(1)                                  # (B, H, W)
    na = Sa.abs().pow(2).sum(1).sqrt()
    nb = Sb.abs().pow(2).sum(1).sqrt()
    rho = ip.abs() / (na * nb).clamp_min(1e-12)
    phi = torch.angle(ip)
    rot = Sa * torch.exp(1j * phi)[:, None]
    e = lambda v: float(v.abs().pow(2).sum(1)[supp].sum())
    nb2 = e(Sb)
    w = Sb.abs() ** 2 * supp[:, None]                             # per-coil weights, on the support
    left = torch.angle(Sb * rot.conj())                           # each coil's leftover phase
    per_coil = ((w * left ** 2).sum((2, 3)) / w.sum((2, 3)).clamp_min(1e-30)).sqrt()[0]
    r = rho[supp].float()
    return dict(raw=(e(Sb - Sa) / nb2) ** 0.5, after_phase=(e(Sb - rot) / nb2) ** 0.5,
                magnitude=(float((Sb.abs() - Sa.abs()).pow(2).sum(1)[supp].sum()) / nb2) ** 0.5,
                rho_med=float(r.median()), rho_p5=float(torch.quantile(r, 0.05)),
                per_coil=per_coil, rho=rho, phi=phi, left=left)


def ref_coil(S):
    # the coil ESPIRiT made real: the smallest energy-weighted |phase| on the support
    w = S.abs() ** 2 * supp[:, None]
    return int(((w * torch.angle(S).abs()).sum((2, 3)) / w.sum((2, 3)).clamp_min(1e-30)).argmin(1)[0])


op_check = compare_maps(S_fs, S_op)
print(f"sanity: the operator's maps vs PDFS maps recomputed here: coherence median "
      f"{op_check['rho_med']:.5f} (differs from 1 only by the measurement noise in y's ACS)")
print(f"the coil each scan's maps are real on (ESPIRiT's strongest coil): PDFS {ref_coil(S_fs)}, "
      f"PD {ref_coil(S_pd)} -- different coils put a shared phase between the two sets of maps")
if ACS_REF == acs:
    print(f"NOTE: ACS_REF = {ACS_REF} is the operator's own count, so the yardstick is 0")

res = {f"same PDFS scan, ACS {acs} vs {ACS_REF}": compare_maps(S_fs, S_fs_ref),
       "PD vs PDFS (same slice)": compare_maps(S_fs, S_pd)}
print(f"\n{'':<32} {'||Sb - Sa||':>11} {'after phase':>11} {'| |Sb|-|Sa| |':>13} "
      f"{'coherence':>9} {'5th pct':>8}")
for k_, d in res.items():
    print(f"{k_:<32} {d['raw']:11.4f} {d['after_phase']:11.4f} {d['magnitude']:13.4f} "
          f"{d['rho_med']:9.5f} {d['rho_p5']:8.5f}")
print("(relative to ||Sb||; 'after phase' removes one phase per pixel shared by every coil)")
for k_, d in res.items():
    print(f"\n{k_}: each coil's leftover phase after the shared one, weighted RMS (degrees)")
    print("  " + " ".join(f"{float(v):5.1f}" for v in torch.rad2deg(d["per_coil"])))

In [ ]:
d = res["PD vs PDFS (same slice)"]
energy = (S_fs.abs() ** 2 * supp[:, None]).sum((2, 3))[0]
coils = energy.argsort(descending=True)[:COILS_SHOW].tolist()
fig, ax = plt.subplots(3, len(coils), figsize=(2.6 * len(coils), 7.8), squeeze=False)
for j, c in enumerate(coils):
    ax[0, j].imshow(show(S_fs[:, c].abs()), cmap="gray", vmin=0, vmax=1)
    ax[1, j].imshow(show(S_pd[:, c].abs()), cmap="gray", vmin=0, vmax=1)
    ax[2, j].imshow(masked_phase(d["left"][:, c], supp), cmap="twilight", vmin=-np.pi, vmax=np.pi)
    ax[0, j].set_title(f"coil {c}", fontsize=9)
for a_ in ax.flat:
    a_.set_xticks([]); a_.set_yticks([])
ax[0, 0].set_ylabel("|PDFS map|", fontsize=8)
ax[1, 0].set_ylabel("|PD map|", fontsize=8)
ax[2, 0].set_ylabel("leftover phase\n(shared phase removed)", fontsize=8)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(9, 3.4))
im = ax[0].imshow(show(d["rho"]), cmap="viridis", vmin=0.9, vmax=1)
ax[0].set_title("coherence rho (0.9 to 1)", fontsize=9); plt.colorbar(im, ax=ax[0], fraction=0.046)
im = ax[1].imshow(masked_phase(d["phi"], supp), cmap="twilight", vmin=-np.pi, vmax=np.pi)
ax[1].set_title("shared phase phi (PD vs PDFS)", fontsize=9); plt.colorbar(im, ax=ax[1], fraction=0.046)
ax[2].imshow(show(supp), cmap="gray"); ax[2].set_title("support", fontsize=9)
for a_ in ax:
    a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

**A consequence to check.** If $S^{\rm PD} = e^{i\varphi} S^{\rm PDFS}$, the guide combined under its own
maps is the guide combined under the PDFS maps, times $e^{-i\varphi}$:
$\sum_c \overline{S^{\rm PD}_c}\,k_c = e^{-i\varphi}\sum_c \overline{S^{\rm PDFS}_c}\,k_c$. Then
switching the guide to the target's maps (`guide_maps = "operator"`) changes the guide's phase and
nothing else.

In [ ]:
g_own = Sense(S_pd).adjoint(gk)
g_fs = Sense(S_fs).adjoint(gk)
pred = g_fs * torch.exp(-1j * d["phi"])[:, None]
w = g_own.abs()[:, 0][supp]
miss = torch.angle(g_own * pred.conj())[:, 0][supp]
print(f"guide on its own maps vs guide on the PDFS maps x e^(-i phi):")
print(f"  magnitude: || |own| - |PDFS maps| || / ||own|| = "
      f"{float((g_own.abs() - g_fs.abs())[:, 0][supp].norm() / w.norm()):.4f}")
print(f"  phase left after e^(-i phi): weighted RMS "
      f"{float(torch.rad2deg(((w ** 2 * miss ** 2).sum() / (w ** 2).sum()).sqrt())):.2f} degrees")

## 2. The phase between the bridge's two ends

For each way of building the pair, the phase difference $\Delta(r) = \arg\big(g(r)\,\overline{x_0(r)}\big)$
on the support, weighted by $|g|\,|x_0|$:

| column | what it measures |
|---|---|
| coherence | $\lvert\sum w\,e^{i\Delta}\rvert / \sum w$: 1 when the two ends differ by one constant phase |
| RMS | the weighted RMS of $\Delta$ itself |
| − constant | after removing the best single phase |
| − smooth | after removing the smooth phase: the phase of $g\bar x_0$ blurred by `LP_SIGMA` px |
| > 90° | the share of the weight where the ends point more than 90° apart |
| cancel | the magnitude lost at the bridge midpoint: $1 - \sum\lvert\tfrac12 x_0 + \tfrac12 g\rvert / \sum(\tfrac12\lvert x_0\rvert + \tfrac12\lvert g\rvert)$ |

The rows are:

1. **as trained**: the target on the operator's maps, the guide on its own maps;
2. **guide on the target's maps** (`guide_maps = "operator"`);
3. **`phase_correct` on both**: each image's low-resolution phase removed by its own calibration
   image, the alignment the retraining would use. It needs only each scan's ACS, so it is available
   at test time.

If the "− smooth" column is far below "RMS", the mismatch is a smooth field, which is what
`phase_correct` removes.

In [ ]:
S_fs_pc, S_pd_pc = maps(kspace, pc=True), maps(g, gmask, lines=gacs, pc=True)
pairs = {"as trained": (x0, x1),
         "guide on the target's maps": (x0, Sense(S_op).adjoint(gk)),
         "phase_correct on both": (Sense(S_fs_pc).adjoint(ifftc(kspace)), Sense(S_pd_pc).adjoint(gk))}


def lowpass(c, s=LP_SIGMA):
    H, W_ = c.shape[-2:]
    fy = torch.fft.fftfreq(H, device=c.device)[:, None]
    fx = torch.fft.fftfreq(W_, device=c.device)[None]
    return torch.fft.ifft2(torch.fft.fft2(c) * torch.exp(-2 * (np.pi * s) ** 2 * (fy ** 2 + fx ** 2)))


def phase_stats(t_img, g_img):
    c = (g_img * t_img.conj())[:, 0]
    w = c.abs()[supp]
    dlt = torch.angle(c)[supp]
    z = (w * torch.exp(1j * dlt)).sum()
    rms = lambda a: float(torch.rad2deg(((w * a ** 2).sum() / w.sum()).sqrt()))
    const = torch.angle(torch.exp(1j * (dlt - torch.angle(z))))
    fine = torch.angle(c * sgn(lowpass(c)).conj())[supp]
    mid = 0.5 * t_img + 0.5 * g_img
    cancel = 1 - float(mid.abs()[:, 0][supp].sum() / (0.5 * t_img.abs() + 0.5 * g_img.abs())[:, 0][supp].sum())
    return dict(R=float(z.abs() / w.sum()), rms=rms(dlt), const=rms(const), fine=rms(fine),
                big=float(w[dlt.abs() > np.pi / 2].sum() / w.sum()), cancel=cancel,
                delta=torch.angle(c), mid=mid)


stats = {k_: phase_stats(*v) for k_, v in pairs.items()}
print(f"{'':<28} {'coherence':>9} {'RMS':>7} {'-const':>7} {'-smooth':>8} {'>90 deg':>8} {'cancel':>7}")
for k_, s_ in stats.items():
    print(f"{k_:<28} {s_['R']:9.3f} {s_['rms']:6.1f}d {s_['const']:6.1f}d {s_['fine']:7.1f}d "
          f"{100 * s_['big']:7.1f}% {100 * s_['cancel']:6.1f}%")

VMAX = q99(x0)
fig, ax = plt.subplots(2, len(stats) + 1, figsize=(2.8 * (len(stats) + 1), 6), squeeze=False)
ax[0, 0].imshow(show(x0.abs()), cmap="gray", vmin=0, vmax=VMAX); ax[0, 0].set_title("|target|", fontsize=9)
ax[1, 0].imshow(show(x1.abs()), cmap="gray", vmin=0, vmax=q99(x1)); ax[1, 0].set_title("|guide|", fontsize=9)
for j, (k_, s_) in enumerate(stats.items(), start=1):
    ax[0, j].imshow(masked_phase(s_["delta"], supp), cmap="twilight", vmin=-np.pi, vmax=np.pi)
    ax[0, j].set_title(f"phase(g x0*)\n{k_}", fontsize=9)
    ax[1, j].imshow(show(s_["mid"].abs()), cmap="gray", vmin=0, vmax=VMAX)
    ax[1, j].set_title("|midpoint (x0 + g) / 2|", fontsize=9)
for a_ in ax.flat:
    a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

## 3. Oracle swap along one sampler run

One I2SB-Recon run (the DDPM posterior, CG on every guess with `DC_LAM`, `DC_ITERS` iterations),
exactly as the intermediates notebook writes it out. At every step the network's guess $\hat x_0$
is also given to CG in two oracle versions. Neither changes the run itself:

- **true phase**: $|\hat x_0|\,\mathrm{sgn}(x_0)$, the guess's magnitude with the target's phase;
- **true magnitude**: $|x_0|\,\mathrm{sgn}(\hat x_0)$, the target's magnitude with the guess's phase.
  This isolates the phase error.

Per step the table gives:

- the guess's **phase error**: the weighted RMS of $\arg(\hat x_0\,\bar x_0)$ on the support;
- the **data residual** $\|Ex - y\|/\|y\|$ before CG;
- the **PSNR after CG** for each version.

If "true phase" collapses the residual and lifts the post-CG PSNR, and "true magnitude" stays
noisy, the phase is what CG is fighting.

The last cell then reruns the whole sampler with each oracle applied at every step before CG.

In [ ]:
mu_x0, mu_x1, std_sb = bridge_coeffs(bridge)
t_of = lambda n: float(mu_x1[n])
SWAPS = {"guess": lambda v: v,
         "true phase": lambda v: v.abs() * sgn(x0),
         "true magnitude": lambda v: x0.abs() * sgn(v)}


def sigma_at(n):
    return forward_std(bridge, torch.full((x1.shape[0],), int(n), device=dev, dtype=torch.long),
                       xdim=x1.shape[1:])


def kappa_of(z):
    return 1.0 if kappa is None else _per_sample(kappa, z.shape[0], z.shape[1:], z.real)


def cg(v):
    return v if DC_LAM is None else cg_data_consistency(v, y, E, DC_LAM, DC_ITERS)


@torch.no_grad()
def run_recon(swap="guess", keep=False):
    trace = []
    with torch.random.fork_rng(devices=[dev] if dev.type == "cuda" else []):
        torch.manual_seed(SEED)
        steps = space_indices(n_steps(bridge), NFE + 1)
        z = x1.clone()
        k = kappa_of(z)
        for n, p in zip(steps[1:][::-1], steps[:-1][::-1]):
            x0_hat = predict_x0_recon(net, z, sigma_at(n), x1=x1, y=y, E=E, sigma_n=sigma_n,
                                      guide=guide)
            if keep:
                trace.append(dict(t=t_of(n), s=t_of(p), x0_hat=x0_hat.cpu()))
            sn2, sp2 = bridge.std_fwd[n] ** 2, bridge.std_fwd[p] ** 2
            a, b = (sn2 - sp2) / sn2, sp2 / sn2
            walked = a * cg(SWAPS[swap](x0_hat)) + b * z
            noise = (k * (sp2 * (sn2 - sp2) / sn2).sqrt() * torch.randn_like(walked) if p > 0
                     else torch.zeros_like(walked))
            z = walked + noise
    return z, trace


def psnr(v):
    return float(compute_metrics(gt.abs(), v.to(gt.device).abs(), data_range=mr)["psnr"])


def resid(v):
    return float((E.forward(v.to(dev)) - y).abs().norm() / y.abs().norm())


def phase_err(v):
    c = (v.to(dev) * x0.conj())[:, 0]
    w = c.abs()[supp]
    return float(torch.rad2deg(((w * torch.angle(c)[supp] ** 2).sum() / w.sum()).sqrt()))


recon, trace = run_recon(keep=True)
rows = []
print(f"{'t':>6} {'phase err':>9} | {'residual before CG':^29} | {'PSNR after CG':^24}")
print(f"{'':>6} {'':>9} | {'guess':>9} {'true ph':>9} {'true mag':>9} | {'guess':>7} {'true ph':>7} "
      f"{'true mag':>8}")
for r in trace:
    v = r["x0_hat"].to(dev)
    sw = {k_: f(v) for k_, f in SWAPS.items()}
    dc = {k_: cg(u) for k_, u in sw.items()}
    row = dict(t=r["t"], perr=phase_err(v), res={k_: resid(u) for k_, u in sw.items()},
               psnr={k_: psnr(u) for k_, u in dc.items()}, dc={k_: u.cpu() for k_, u in dc.items()})
    rows.append(row)
    print(f"{r['t']:6.3f} {row['perr']:8.1f}d | " + " ".join(f"{row['res'][k_]:9.4f}" for k_ in SWAPS)
          + " | " + " ".join(f"{row['psnr'][k_]:7.2f}" for k_ in ("guess", "true phase"))
          + f" {row['psnr']['true magnitude']:8.2f}")

ts = [r["t"] for r in rows]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(ts, [r["perr"] for r in rows], "-o", ms=3, color="k")
ax[0].set_ylabel("phase error of the guess (deg)")
for k_, c_ in zip(SWAPS, ("tab:red", "tab:blue", "tab:green")):
    ax[1].semilogy(ts, [r["res"][k_] for r in rows], "-o", ms=3, color=c_, label=k_)
    ax[2].plot(ts, [r["psnr"][k_] for r in rows], "-o", ms=3, color=c_, label=k_)
ax[1].set_ylabel("||Ex - y|| / ||y||, before CG")
ax[2].set_ylabel("PSNR after CG (dB)")
for a_ in ax:
    a_.invert_xaxis(); a_.set_xlabel("bridge time t"); a_.grid(alpha=0.3)
ax[1].legend(fontsize=8); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
picks = sorted({int(round(i)) for i in np.linspace(0, len(rows) - 1, SHOW)})
ERR = 0.2 * VMAX
labels = [("phase error of the guess", None)] + [(f"| |after CG| - |target| |\n{k_}", k_)
                                                  for k_ in SWAPS]
fig, ax = plt.subplots(len(labels), len(picks), figsize=(2.4 * len(picks), 2.3 * len(labels)),
                       squeeze=False)
for j, i in enumerate(picks):
    r = rows[i]
    perr = torch.angle(trace[i]["x0_hat"].to(dev) * x0.conj())
    ax[0, j].imshow(masked_phase(perr, supp), cmap="twilight", vmin=-np.pi, vmax=np.pi)
    ax[0, j].set_title(f"t = {r['t']:.2f}", fontsize=9)
    for k, (_, key) in enumerate(labels[1:], start=1):
        ax[k, j].imshow(show((r["dc"][key].to(dev).abs() - gt.abs()).abs()), cmap="magma",
                        vmin=0, vmax=ERR)
for k, (lab, _) in enumerate(labels):
    ax[k, 0].set_ylabel(lab, fontsize=8)
for a_ in ax.flat:
    a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
finals = {"guess (the real run)": recon}
for k_ in ("true phase", "true magnitude"):
    finals[f"oracle {k_}"] = run_recon(k_)[0]
print(f"{'whole run, CG on':<28} {'PSNR':>7} {'SSIM':>7} {'residual':>9}")
for k_, v in finals.items():
    m = compute_metrics(gt.abs(), v.to(gt.device).abs(), data_range=mr)
    print(f"{k_:<28} {float(m['psnr']):7.2f} {float(m['ssim']):7.4f} {resid(v):9.4f}")

fig, ax = plt.subplots(2, len(finals) + 1, figsize=(2.8 * (len(finals) + 1), 5.8), squeeze=False)
ax[0, 0].imshow(show(gt.abs()), cmap="gray", vmin=0, vmax=VMAX); ax[0, 0].set_title("target", fontsize=9)
ax[1, 0].axis("off")
for j, (k_, v) in enumerate(finals.items(), start=1):
    ax[0, j].imshow(show(v.abs()), cmap="gray", vmin=0, vmax=VMAX); ax[0, j].set_title(k_, fontsize=9)
    ax[1, j].imshow(show((v.to(dev).abs() - gt.abs()).abs()), cmap="magma", vmin=0, vmax=ERR)
for a_ in ax.flat:
    a_.set_xticks([]); a_.set_yticks([])
ax[1, 1].set_ylabel("| |x| - |target| |", fontsize=8)
plt.tight_layout(); plt.show()

## Reading the results

- **Part 1, maps.** If PD vs PDFS sits at the yardstick's level after the shared phase is removed,
  the maps differ only in phase. Leftover structure in the per-coil phase panels, or a coherence
  well below the yardstick's, means the coil profiles changed between the scans (motion, or a slice
  pairing off by one). Part 2's alignments can't fix that.
- **Part 2, the two ends.** The "as trained" row is the mismatch the network learned to bridge. If
  the target's maps or `phase_correct` remove most of it, and most of what is left is smooth, then
  retraining with `phase_correct` on both sets of maps (`mri.online_smaps_kws.phase_correct`) is
  the fix. The guided-recon config generator does not set that option yet.
- **Part 3, the oracle.** A large gap between "guess" and "true phase", in the residual and in the
  post-CG PSNR, says the phase is the error CG is fighting. If "true phase" barely helps, the
  instability comes from somewhere else, such as the magnitude or the CG weight.